In [0]:
# ===================================================
# BLOCK 1 — PIPELINE IMPORTS AND CONSTANTS (PYTHON)
# ===================================================

"""
Define the governed table names, source location, and accepted test-result
values used by the streaming pipeline.
"""

from pyspark import pipelines as dp
from pyspark.sql import functions as F
from pyspark.sql import types as T

CATALOG = "semiconplus_portfolio"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
QUARANTINE_SCHEMA = "quarantine"

SOURCE_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/external_source/streaming_demo/input"
)
SCHEMA_DIRECTORY = (
    "/Volumes/semiconplus_portfolio/landing/external_source/"
    "_schemas/sdp_streaming/bronze_test_results"
)

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.streaming_test_results"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.streaming_test_results"
QUARANTINE_TABLE = (
    f"{CATALOG}.{QUARANTINE_SCHEMA}.streaming_test_results"
)
LATE_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.streaming_late_test_results"
GOLD_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.mart_streaming_yield_5m"

VALID_EVENT_TYPES = ["TEST_RESULT"]
VALID_STATUSES = ["PASS", "FAIL", "ALARM"]

In [0]:
# ===================================================
# BLOCK 2 — BRONZE STREAMING INGESTION (PYTHON)
# ===================================================

"""
Incrementally ingest JSON event files with Auto Loader while retaining rescued
content and source metadata for replay, lineage, and incident investigation.
Pipeline-managed state tracks processed files between triggered updates.
"""

@dp.table(
    name=BRONZE_TABLE,
    comment=(
        "Raw simulated test-result events ingested incrementally from the "
        "SemiconPlus external landing volume."
    ),
    table_properties={
        "quality": "bronze",
        "pipelines.autoOptimize.managed": "true",
    },
)
def bronze_streaming_test_results():
    return (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", SCHEMA_DIRECTORY)
        .option("cloudFiles.schemaEvolutionMode", "rescue")
        .option("rescuedDataColumn", "_rescued_data")
        .option("cloudFiles.inferColumnTypes", "false")
        .load(SOURCE_DIRECTORY)
        .select(
            "*",
            F.col("_metadata.file_path").alias("_source_file_path"),
            F.col("_metadata.file_name").alias("_source_file_name"),
            F.col("_metadata.file_modification_time").alias(
                "_source_file_modification_time"
            ),
            F.current_timestamp().alias("_ingested_at_utc"),
        )
    )


In [0]:
# ===================================================
# BLOCK 3 — PARSED EVENT VIEW (PYTHON)
# ===================================================

"""
Standardize identifiers, cast timestamps and test time, and assign all
deterministic validation failures before accepted and quarantined records are
routed to separate governed tables.
"""

@dp.temporary_view(name="streaming_test_results_classified")
def streaming_test_results_classified():
    source_df = spark.readStream.table(BRONZE_TABLE)

    standardized_df = source_df.select(
        F.upper(F.trim("device_id")).alias("device_id"),
        F.upper(F.trim("equipment_id")).alias("equipment_id"),
        F.trim("event_id").alias("event_id"),
        F.expr("try_cast(event_timestamp_utc AS timestamp)").alias(
            "event_timestamp_utc"
        ),
        F.expr("try_cast(ingestion_timestamp_utc AS timestamp)").alias(
            "ingestion_timestamp_utc"
        ),
        F.expr("try_cast(is_deliberately_late AS boolean)").alias(
            "is_deliberately_late"
        ),
        F.upper(F.trim("product_group_id")).alias("product_group_id"),
        F.upper(F.trim("site_id")).alias("site_id"),
        F.upper(F.trim("event_type")).alias("event_type"),
        F.upper(F.trim("status")).alias("status"),
        F.expr("try_cast(test_time_seconds AS double)").alias(
            "test_time_seconds"
        ),
        "_rescued_data",
        "_source_file_path",
        "_source_file_name",
        "_source_file_modification_time",
        "_ingested_at_utc",
    )

    return standardized_df.withColumn(
        "_quality_reasons",
        F.array_compact(
            F.array(
                F.when(
                    F.col("event_id").isNull() | (F.col("event_id") == ""),
                    F.lit("MISSING_EVENT_ID"),
                ),
                F.when(
                    F.col("event_timestamp_utc").isNull(),
                    F.lit("INVALID_EVENT_TIMESTAMP"),
                ),
                F.when(
                    F.col("device_id").isNull() | (F.col("device_id") == ""),
                    F.lit("MISSING_DEVICE_ID"),
                ),
                F.when(
                    F.col("equipment_id").isNull()
                    | (F.col("equipment_id") == ""),
                    F.lit("MISSING_EQUIPMENT_ID"),
                ),
                F.when(
                    ~F.col("event_type").isin(VALID_EVENT_TYPES),
                    F.lit("INVALID_EVENT_TYPE"),
                ),
                F.when(
                    ~F.col("status").isin(VALID_STATUSES),
                    F.lit("INVALID_STATUS"),
                ),
                F.when(
                    F.col("test_time_seconds").isNull()
                    | (F.col("test_time_seconds") <= 0),
                    F.lit("INVALID_TEST_TIME_SECONDS"),
                ),
                F.when(
                    F.col("ingestion_timestamp_utc").isNull(),
                    F.lit("INVALID_INGESTION_TIMESTAMP"),
                ),
                F.when(
                    F.col("is_deliberately_late").isNull(),
                    F.lit("INVALID_LATE_ARRIVAL_FLAG"),
                ),
                F.when(
                    F.col("product_group_id").isNull()
                    | (F.col("product_group_id") == ""),
                    F.lit("MISSING_PRODUCT_GROUP_ID"),
                ),
                F.when(
                    F.col("site_id").isNull() | (F.col("site_id") == ""),
                    F.lit("MISSING_SITE_ID"),
                ),
                F.when(
                    F.col("_rescued_data").isNotNull(),
                    F.lit("RESCUED_SOURCE_DATA"),
                ),
            )
        ),
    )


In [0]:
# ===================================================
# BLOCK 4 — SILVER ACCEPTED STREAM (PYTHON)
# ===================================================

"""
Retain valid on-time test results, apply a two-hour event-time watermark, and
deduplicate event identifiers with bounded state. Deliberately late records are
routed separately so their handling remains observable and testable.
"""

@dp.table(
    name=SILVER_TABLE,
    comment=(
        "Validated and watermark-deduplicated simulated test-result events."
    ),
    table_properties={"quality": "silver"},
)
@dp.expect_or_drop(
    "valid_on_time_business_record",
    "size(_quality_reasons) = 0 AND is_deliberately_late = false",
)
def silver_streaming_test_results():
    return (
        spark.readStream.table("streaming_test_results_classified")
        .filter(
            (F.size("_quality_reasons") == 0)
            & (F.col("is_deliberately_late") == F.lit(False))
        )
        .withWatermark("event_timestamp_utc", "2 hours")
        .dropDuplicatesWithinWatermark(["event_id"])
        .withColumn("_processed_at_utc", F.current_timestamp())
    )

In [0]:
# ===================================================
# BLOCK 5 — DELIBERATELY LATE EVENT ROUTING (PYTHON)
# ===================================================

"""
Persist valid events intentionally generated behind the accepted lateness
threshold. Separating them before the accepted stream preserves evidence of
late-data handling rather than allowing the watermark to discard them silently.
"""

@dp.table(
    name=LATE_TABLE,
    comment="Valid simulated test results routed for deliberate late arrival.",
    table_properties={"quality": "silver_late"},
)
def streaming_late_test_results():
    return (
        spark.readStream.table("streaming_test_results_classified")
        .filter(
            (F.size("_quality_reasons") == 0)
            & (F.col("is_deliberately_late") == F.lit(True))
        )
        .withWatermark("ingestion_timestamp_utc", "2 hours")
        .dropDuplicatesWithinWatermark(["event_id"])
        .withColumn(
            "arrival_delay_seconds",
            F.unix_timestamp("ingestion_timestamp_utc")
            - F.unix_timestamp("event_timestamp_utc"),
        )
        .withColumn("_routed_at_utc", F.current_timestamp())
    )


In [0]:
# ===================================================
# BLOCK 6 — STREAMING QUARANTINE (PYTHON)
# ===================================================

"""
Persist malformed and contract-violating events with their failure reasons and
source lineage so rejected data can be investigated and replayed after repair.
"""

@dp.table(
    name=QUARANTINE_TABLE,
    comment=(
        "Rejected simulated test-result events with quality reasons and lineage."
    ),
    table_properties={"quality": "quarantine"},
)
def quarantine_streaming_test_results():
    return (
        spark.readStream.table("streaming_test_results_classified")
        .filter(F.size("_quality_reasons") > 0)
        .withColumn("_quarantined_at_utc", F.current_timestamp())
    )


In [0]:
# ===================================================
# BLOCK 7 — GOLD FIVE-MINUTE YIELD MART (PYTHON)
# ===================================================

"""
Aggregate accepted streaming test results into five-minute operational windows
for near-real-time yield and alarm monitoring without exposing event-level data.
The materialized view is refreshed during each bounded pipeline update.
"""

@dp.materialized_view(
    name=GOLD_TABLE,
    comment=(
        "Five-minute yield and alarm metrics generated from validated "
        "simulated streaming test results."
    ),
    table_properties={"quality": "gold"},
)
def mart_streaming_yield_5m():
    return (
        spark.read.table(SILVER_TABLE)
        .groupBy(
            F.window("event_timestamp_utc", "5 minutes").alias(
                "event_window"
            ),
            "site_id",
            "equipment_id",
            "product_group_id",
            "device_id",
        )
        .agg(
            F.count("*").alias("event_count"),
            F.sum(
                F.when(F.col("status") == "PASS", F.lit(1)).otherwise(F.lit(0))
            ).alias("pass_count"),
            F.sum(
                F.when(F.col("status") == "FAIL", F.lit(1)).otherwise(F.lit(0))
            ).alias("fail_count"),
            F.sum(
                F.when(F.col("status") == "ALARM", F.lit(1)).otherwise(F.lit(0))
            ).alias("alarm_count"),
            F.avg("test_time_seconds").alias("average_test_time_seconds"),
            F.max("test_time_seconds").alias("maximum_test_time_seconds"),
            F.max("event_timestamp_utc").alias("latest_event_timestamp_utc"),
        )
        .select(
            F.col("event_window.start").alias("window_start_utc"),
            F.col("event_window.end").alias("window_end_utc"),
            "site_id",
            "equipment_id",
            "product_group_id",
            "device_id",
            "event_count",
            "pass_count",
            "fail_count",
            "alarm_count",
            F.when(
                (F.col("pass_count") + F.col("fail_count")) > 0,
                F.col("pass_count")
                / (F.col("pass_count") + F.col("fail_count")),
            ).alias("yield_rate"),
            "average_test_time_seconds",
            "maximum_test_time_seconds",
            "latest_event_timestamp_utc",
            F.current_timestamp().alias("_refreshed_at_utc"),
        )
    )